In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
import pickle as pkl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.plotting_tools.Bins import bins
from src.assets.lumi import lumi_dict
from src.plotting_tools.Bins import Bins
from src.assets.output_dir import output_dir
from src.plotting_tools.cms_format import cms_format_fig, cms_style

outdir = output_dir

In [ ]:
cms_style()

In [ ]:
era = '2017'
lumi_fraction = lumi_dict[str(era)]/lumi_dict['201X']
lumi_fraction
xrange = (120,401)

In [ ]:
with open('{}/data/{}_bff_interp_dbs_norm.pkl'.format(outdir, era), 'rb') as f:
    data = pkl.load(f)
masses = data.mass.unique()

In [ ]:
##
## Make acceptance details
##

In [ ]:
accpt_df = pd.read_csv('/eos/cms/store/group/phys_exotica/bffZprime/assets_june_23'+"/data_gen_b_s/summary_df.csv")
accpt_df

isrfsr = abs((accpt_df['Weight_ISRFSR_Up']-accpt_df['Weight_ISRFSR_Down']))/(accpt_df['acceptance']*2)

pdf = abs(accpt_df['Weight_PDF_Up']-accpt_df['Weight_PDF_Down'])/(accpt_df['acceptance']*2)

min(isrfsr), max(isrfsr), np.mean(isrfsr), min(pdf), max(pdf), np.mean(pdf)


In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, RationalQuadratic

In [ ]:
def GPR_fit_and_predict(x,y,std, xp, plot=False, length_scale_bounds=(100,10000),alpha=1e-1):
    kernel = 1 *  RBF(length_scale=100, length_scale_bounds=length_scale_bounds)
    gaussian_process = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9, random_state=0, alpha=alpha, normalize_y=True)
    gaussian_process.fit(x, y)
    #predict
    mean_prediction, std_prediction = gaussian_process.predict(xp, return_std=True)
    mean_prediction = mean_prediction.reshape(-1)
    if plot:
        plot.errorbar(x, y, yerr=std, label='Data')
        #gpr:
        plot.plot(xp, mean_prediction, color='orange', label='GPR Fit')
        plot.fill_between(xp.reshape(-1),  mean_prediction+std_prediction, mean_prediction-std_prediction, alpha=.5, color='orange')
    return  mean_prediction, std_prediction

In [ ]:
def extend_accpt_df(reg, sig_type):
    tadf = accpt_df[(accpt_df.reg==reg) & (accpt_df.type==sig_type)]

    #accp
    fig, ax = plt.subplots()
    cms_format_fig(era, ax, "\emph{Simulation}")
    ax.set_ylabel('Acceptance')
    ax.set_xlabel('$m_{Z\prime}$ [GeV]')
    ax.legend(title=f'{reg} {sig_type}')
    accep, stat = GPR_fit_and_predict(tadf.mass.to_numpy().reshape(-1, 1), tadf.acceptance.to_numpy().reshape(-1, 1), tadf.statistical.to_numpy(),
       masses.reshape(-1,1), plot=ax, length_scale_bounds=(100,10000), alpha=tadf.statistical.to_numpy()*1e1)
    
    
    fig, ax = plt.subplots()
    cms_format_fig(era, ax, "\emph{Simulation}")
    ax.set_ylabel('ISR/FSR Weight')
    ax.set_xlabel('$m_{Z\prime}$ [GeV]')
    ax.legend(title=f'{reg} {sig_type}')
    Weight_ISRFSR_Up, _  = GPR_fit_and_predict(tadf.mass.to_numpy().reshape(-1, 1), tadf.Weight_ISRFSR_Up.to_numpy().reshape(-1, 1), 0*tadf.statistical.to_numpy(),
       masses.reshape(-1,1), plot=ax, length_scale_bounds=(100,10000), alpha=2e-1)
    
    Weight_ISRFSR_Down, _  = GPR_fit_and_predict(tadf.mass.to_numpy().reshape(-1, 1), tadf.Weight_ISRFSR_Down.to_numpy().reshape(-1, 1), 0*tadf.statistical.to_numpy(),
       masses.reshape(-1,1), plot=ax, length_scale_bounds=(100,10000), alpha=2e-1)
    
    fig, ax = plt.subplots()
    cms_format_fig(era, ax, "\emph{Simulation}")
    ax.set_ylabel('PDF Weight')
    ax.set_xlabel('$m_{Z\prime}$ [GeV]')
    ax.legend(title=f'{reg} {sig_type}')
    Weight_PDF_Up, _  = GPR_fit_and_predict(tadf.mass.to_numpy().reshape(-1, 1), tadf.Weight_PDF_Up.to_numpy().reshape(-1, 1), 0*tadf.statistical.to_numpy(),
       masses.reshape(-1,1), plot=ax, length_scale_bounds=(100,10000), alpha=2e-1)

    Weight_PDF_Down, _  = GPR_fit_and_predict(tadf.mass.to_numpy().reshape(-1, 1), tadf.Weight_PDF_Down.to_numpy().reshape(-1, 1), 0*tadf.statistical.to_numpy(),
       masses.reshape(-1,1), plot=ax, length_scale_bounds=(100,10000), alpha=2e-1)
    
    #make df
    df_list = []
    for mass, a, s, ISRFSR_Up, ISRFSR_Down, PDF_up, PDF_Down in zip(masses, accep, stat, Weight_ISRFSR_Up, Weight_ISRFSR_Down, Weight_PDF_Up, Weight_PDF_Down):
        df_list.append({"mass": mass, "reg": reg, "type": sig_type, 
                        "acceptance": a, 'statistical': s, 
                        "Weight_ISRFSR_Up": ISRFSR_Up, "Weight_ISRFSR_Down": ISRFSR_Down,
                        "Weight_PDF_Up": PDF_up, "Weight_PDF_Down": PDF_Down})
    return df_list

In [ ]:
df_list = []
for reg in ['SR1', 'SR2']:
    for sig_type in accpt_df.type.unique():
        df_list+= extend_accpt_df(reg, sig_type)

In [ ]:
accpt_df_interpolated = pd.DataFrame(df_list)
accpt_df_interpolated

In [ ]:
for reg in accpt_df_interpolated.reg.unique():
    fig, ax = plt.subplots()
    cms_format_fig('Run 2', ax, "\emph{Simulation}")
    ax.set_ylabel('Acceptance')
    ax.set_xlabel('$m_{Z\prime}$ [GeV]')
    ax.set_ylim(0, .3)
    
    for fstype in accpt_df_interpolated.type.unique():
        if fstype=='shape': continue
        if fstype=='2s': continue
        tdf_int = accpt_df_interpolated[(accpt_df_interpolated.type==fstype) & (accpt_df_interpolated.reg==reg)]
        tdf = accpt_df[(accpt_df.type==fstype) & (accpt_df.reg==reg)]
        
        ax.plot(tdf_int.mass, tdf_int.acceptance, label=fstype)
        ax.fill_between(tdf_int.mass,  tdf_int.acceptance+tdf_int.statistical, tdf_int.acceptance-tdf_int.statistical, alpha=.5)
        #get color
        color = plt.gca().lines[-1].get_color()
        ax.errorbar(tdf.mass, tdf.acceptance, yerr=tdf.statistical, color=color, linestyle='none', marker='o')
    plt.legend()
    plt.show()
    fig.savefig('{}/gen_b_s/accept_int_{}_{}.pdf'.format(outdir, reg, 'Run_2'))
    plt.clf()

In [ ]:
for reg in accpt_df_interpolated.reg.unique():
    fig, ax = plt.subplots(constrained_layout=True)
    cms_format_fig('Run 2', ax, "\emph{Simulation}")
    ax.set_ylabel('ISR/FSR Weight')
    ax.set_xlabel('$m_{Z\prime}$ [GeV]')
    ax.set_ylim(-.01, .01)
    for fstype in accpt_df_interpolated.type.unique():
        if fstype=='shape': continue
        if fstype=='2s': continue
        tdf_int = accpt_df_interpolated[(accpt_df_interpolated.type==fstype) & (accpt_df_interpolated.reg==reg)]
        tdf = accpt_df[(accpt_df.type==fstype) & (accpt_df.reg==reg)]
        
        ax.plot(tdf_int.mass, tdf_int.Weight_ISRFSR_Up, label=fstype)
        color = plt.gca().lines[-1].get_color()
        ax.plot(tdf_int.mass, tdf_int.Weight_ISRFSR_Down, color=color)
        
        ax.scatter(tdf.mass, tdf.Weight_ISRFSR_Up, color=color)
        ax.scatter(tdf.mass, tdf.Weight_ISRFSR_Down, color=color)
    plt.legend(ncol=2)
    plt.show()
    fig.savefig('{}/gen_b_s/ISRFSR_int_{}_{}.pdf'.format(outdir, reg, 'Run_2'))
    plt.clf()

In [ ]:
for reg in accpt_df_interpolated.reg.unique():
    fig, ax = plt.subplots(constrained_layout=True)
    cms_format_fig('Run 2', ax, "\emph{Simulation}")
    ax.set_ylabel('PDF Weight')
    ax.set_xlabel('$m_{Z\prime}$ [GeV]')
    ax.set_ylim(-.0005, .0005)
    for fstype in accpt_df_interpolated.type.unique():
        if fstype=='shape': continue
        if fstype=='2s': continue
        tdf_int = accpt_df_interpolated[(accpt_df_interpolated.type==fstype) & (accpt_df_interpolated.reg==reg)]
        tdf = accpt_df[(accpt_df.type==fstype) & (accpt_df.reg==reg)]
        
        ax.plot(tdf_int.mass, tdf_int.Weight_PDF_Up, label=fstype)
        color = plt.gca().lines[-1].get_color()
        ax.plot(tdf_int.mass, tdf_int.Weight_PDF_Down, color=color)
        
        ax.scatter(tdf.mass, tdf.Weight_PDF_Up, color=color)
        ax.scatter(tdf.mass, tdf.Weight_PDF_Down, color=color)
    plt.legend(ncol=2)
    plt.show()
    fig.savefig('{}/gen_b_s/PDF_int_{}_{}.pdf'.format(outdir, reg, 'Run_2'))
    plt.clf()

In [ ]:
##
## rebin_signals
##

In [ ]:
from time import perf_counter

In [ ]:
def rebin_integrate(x,y, xnew_binedges, keep_norm=False, usequad=False, straight_interpolate = False, sample_density=100):
    from scipy.interpolate import interp1d
    from scipy.integrate import quad
    f = interp1d(x,y, fill_value=0, bounds_error=False)
     
    centers = np.array(Bins(bin_edges).calc_bin_centers())
    widths = np.array(Bins(bin_edges).calc_bin_widths())
    if usequad:
        ynew_int = []
        for i, center in enumerate(centers):
            integral = quad(f, xnew_binedges[i], xnew_binedges[i+1])[0]
            ynew_int.append(integral)
        ynew_int = np.array(ynew_int)
    elif straight_interpolate:
        ynew_int = f(centers)*widths
    else:
        ynew_int = []
        for i, (center, width) in enumerate(zip(centers, widths)):
            sample_points = np.linspace(xnew_binedges[i], xnew_binedges[i+1],int(sample_density*width))
            integral = f(sample_points).sum()/sample_density
            ynew_int.append(integral)
        ynew_int = np.array(ynew_int)
        
    
    
    if keep_norm:  
        y_total = y[(x>=np.min(bin_edges)) & (x<=np.max(bin_edges))].sum()
        return centers, ynew_int/ynew_int.sum()*y_total
    return centers, ynew_int

In [ ]:
bin_edges = bins.bin_edges
bin_edges = bin_edges[(bin_edges> xrange[0]) & (bin_edges < xrange[1])]
bin_centers = Bins(bin_edges).calc_bin_centers()
bin_edges, bin_centers

In [ ]:
rebinned = []
start = perf_counter()
for i, row in data.iterrows():
    
    x, y = row.x, row.y
    xnew, ynew = rebin_integrate(x, y, bin_edges, keep_norm=True, sample_density=2)
    rebinned.append(ynew)
    if i%1000==10: 
        end = perf_counter()
        elapsed_time_min = (end-start)/60
        time_per = elapsed_time_min/i
        nremaining = len(data)-i
        time_remaining = time_per*nremaining
        print("ratio done:{:.2f} elapsed minutes: {:.1f} est time remaining: {:.1f}".format(i/len(data), elapsed_time_min, time_remaining))
        
end = perf_counter()
print(end-start)
data['rebinned'] = rebinned

In [ ]:
data

In [ ]:
def make_list(_list, channel, process, systematic, values, masses, norm):
    values = values/values.sum()*norm
    for i, (x, m) in enumerate(zip(values, masses)):
        _list.append({'channel': channel, 'process': process,
                      'systematic': systematic, 'bin': m, 'sum_w': x, 'sum_ww':x})

In [ ]:
csvname = f'{output_dir}/combine_data/{era}/{era}_signal_shapes_df_input.csv'
csvname

In [ ]:
data.mass.unique()

In [ ]:
from src.assets.shape_scaling import scale

In [ ]:
final_state = 'shape'
csv_list = []
acceptance = 1

lumi_fraction = lumi_dict[str(era)]/lumi_dict['201X']



for i, row in data.iterrows():
    if row.dbs!=0.5:continue
    if row.mass!= 125:continue
    x = row.rebinned
    sys = row.sys
    reg = row.reg
    dbs = row.dbs
    mass = row.mass
    channel = reg
    process = f'{mass}_{dbs}_{final_state}'
    systematic = row.sys if row.sys!= 'nom' else 'nominal'
    values = row.rebinned
    
    norm =  scale*acceptance*lumi_fraction
    make_list(csv_list, channel, process, systematic, values, bin_centers, norm)
    break

In [ ]:
df = pd.DataFrame(csv_list)

df.to_csv(csvname, index=False)

In [ ]:
##
## make datacards
##